# Hybrid Models for Hyperspectral Image Classification
**Author:** Valerio Massimo Carioti (Student ID 1983063)

## Project Aim

The goal of the project is to implement and evaluate the CTA-net architecture, a deep learning model that combines convolutional and attention based networks in order to classify pixels of hyperspectral images when only a limited amount of samples is available. It also aims to make a comparison between static data amplification and online augmentation in the same setting.

**Reference Paper:** *CNN-Transformer and Channel-Spatial Attention based network for
hyperspectral image classification with few samples*; C. Fu, T. Zhou, T. Guo, Q. Zhu, F. Luo, B. Du; 2025

## Theoretical Background and Key Concepts

### Hyperspectral Imaging

Hyperspectral Imaging (HSI) captures hundreds of continuous spectral bands for each pixels, coverning a wider range with higher precision with respect to traditional photography (that only considers visible light and splits it in three bands: red, green and blue).
One of the main difficulties encountered when dealing with HSI data is the scarcity of labelled samples. This, combined with the much higher number of informations available for each pixel, makes the usage of large networks unfeasable as they would immediately overfit.

### Pixel Classification With Hyperspectral Images

Due to the fact that a dataset usually consists of a single image, image-to-image pixel classification is not possible. Also, doing it with patches would result in data-leakage. For this resons, pixel classification is obtained by extracting its surrounding patch and feeding it to the network. The same label as the central pixel is then given to the whole patch, making it an image classification problem.

### Local Features and Global Context

To improve pixel classification, a network should capture local fetures while taking in account the global context. This can be achieved by combining convolutional and attention layers.

### Online Data Augmentation

Online data augmentation allows the model to be trained on different samples each epoch, by applying a partially random transformation to the original ones directly when they are feed to the model, instead than before starting the training. By increasing the number of different samples seen by the model, the hope is to mitigate the restrictions given by having a small dataset.

## Implementation Details

### Datasets

#### Pavia University

This dataset is widely used as a benchmark for hyperspectral image classification. It consists of an image taken by the ROSIS sensor (Reflective Optics System Imaging Spectrometer) flying over Pavia. It contains 610x340 pixels with a resolution of 1.3m and 115 spectral bands in a range of 430-860nm wavelenght. 12 of those bands were removed due to being too noisy.

The classes contained in the dataset are divided as follows:
| Label | Class | Samples |
|-|-|-|
| 0 | Undefined | 164624 |
| 1 | Asphalt | 6631 |
| 2 | Meadows | 18649 |
| 3 | Gravel | 2099 |
| 4 | Trees | 3064 |
| 5 | Painted metal sheets | 1345 |
| 6 | Bare Soil | 5029 |
| 7 | Bitumen | 1330 |
| 8 | Self-Blocking Bricks | 3682 |
| 9 | Shadows | 947 |

![Pavia University dataset. (a) False-color map; (b) Ground-truth map.](https://github.com/JoJohnny0/nn-project/blob/main/images/PaviaUniversity.png?raw=true)

#### Indian Pines

This dataset consists of an image collected by Purdue University Research Repository (PURR) in 1992 using NASA's AVIRIS sensor (Airborne Visible/InfraRed Imaging Spectrometer) flying over the Indian Pines test site in North West Indiana. It contains 145x145 pixels with a resolution of 20 meters and 224 spectral bands in a range of 400–2500nm wavelength. 24 of those bands were removed due to being too noisy.

The classes contained in the dataset are divided as follows:
| Label | Class | Samples |
|-|-|-|
| 0 | Undefined | 10776 |
| 1 | Alfalfa | 46 |
| 2 | Corn-Notill | 1428 |
| 3 | Corn-Mintill | 830 |
| 4 | Corn | 237 |
| 5  | Grass-Pasture | 483 |
| 6 | Grass-Trees | 730 |
| 7 | Grass-Pasture-Mowed | 28 |
| 8 | Hay-Windrowed | 478 |
| 9 | Oats | 20 |
| 10 | Soybean-Notill | 972 |
| 11 | Soybean-Mintill | 2455 |
| 12 | Soybean-Clean | 593 |
| 13 | Wheat | 205 |
| 14 | Woods | 1265 |
| 15 | Buildings-Grass-Trees-Drives | 386 |
| 16 | Stone-Steel-Towers | 93 |

![Indian Pines dataset. (a) False-color map; (b) Ground-truth map](https://github.com/JoJohnny0/nn-project/blob/main/images/IndianPines.png?raw=true)

This dataset provides a more difficult challenge than Pavia University's due to the higher number of classes and the fact that they are far more unbalanced and similar to each other.

### Sample Amplification

In order to increase the number of training samples, three tecniques can be applied.
- **Random Rotation:** each image is randomly rotated, using mirror padding.
- **Random Noise:** random noise is added to the image, excluding the central pixels.
- **Averaging Images:** taken two patches of the same class, average them.

These new samples are collected before the training phase and added to the training set, together with the original ones.

### Online Data Augmentation

As an alternative to static sample amplification, also online data augmentation can be applied.

Some of the previous tecniques have been slightly changed:
- Random Noise: instead of adding a random noise with fixed standard deviation, this parameter is casually generated each time in a range from 0 to `sigma`.
- Mixup: to introduce more variance, a linear combination of two images is used instead of the plain average.

Each one of the three transformation has a 1/2 probability to be applied independently from the others.

### Model

The implemented model is a CTA-net and it consists of two main blocks: the mixed Convolution-Transformer (CT) block and the Attention block. Their main features are described below.

![CTA-net architecture](https://github.com/JoJohnny0/nn-project/blob/main/images/cta-net.png?raw=true)

#### CT Block

The mixed Convolution-Transformer block presents two parallel branches: one containing a multi-resolution CNN and one implementing a Conformer-inspired transformer. The outputs of the two branches are concatenated and a residual connection is applied.

The multi-head self attention uses a relative positional encoding that consists of bias applied directly to the attention weights. This bias is learnable and based on the relative position of the features in the input.

![Mixed Convolution-Attention block](https://github.com/JoJohnny0/nn-project/blob/main/images/ct-block.png?raw=true)

![trasnformer](https://github.com/JoJohnny0/nn-project/blob/main/images/transformer.png?raw=true)

#### Attention Block

The attention block is divided into a channel attention block and a spatial attention block.
The first one learns for each channel a weight between 0 and 1 and uses it to multiply the input before applying a residual connection, while the second one does the same with spatial informations.

Those spatial informations are minimum, maximum, average and standard deviation of each pixel over the channels. They are then passed through a convolutional layer and a PReLU activation function before being concatenated with the original input (all this happens in the Spatial information and Concat blocks of the figure below).

![attention](https://github.com/JoJohnny0/nn-project/blob/main/images/attention%20block.png?raw=true)

### Experimental Setup

In order to test how the model would perform in a setting with very few available samples, only 15 samples per class have been used (10 for training and 5 for validation).

The hyperparameters are, generally, the ones suggested in the reference paper and no further tuning has been performed. The number of training epochs when using online data augmentation is higher to compensate to the fact that each epoch has 1/4 of the samples with respect to static sample amplification.

A list of the hyperparameters used follows:

|Hyperparameter | Value |
|-|-|
| Side of each patch | 15 |
| (Max.) std. dev. of the added noise | 0.1 |
| Side of the zero-noise region | 3 |
| # Hidden channels | 128 |
| # Heads | 2 |
| Dropout | 0.1 |
| Learning rate | 8e-5 |
| Batch size | 32 |
| Training Epochs (static amplification) | 150 |
| Training Epochs (online augmentation) | 200 |

The code has been executed on Google Colab using the T4 GPU.

## Results

The following tables show the obtained results on the two different datasets.

**Results on Pavia University HSI dataset**

![Pavia University dataset validation loss](https://github.com/JoJohnny0/nn-project/blob/main/images/PaviaUniversity%20Loss.png?raw=true)

| metric | Static Amplification | Online Data Augmentation | Δ% |
|-|-|-|-|
| Overall Accuracy | 86.98 ± 4.94 | 86.57 ± 4.65 | -0.47% |
| Average Accuracy | 90.72 ± 2.55 | 90.81 ± 2.35 | +0.10% |
| Kappa | 83.33 ± 5.94 | 82.81 ± 5.66 | -0.63% |
| Class 1 Accuracy | 84.46 ± 8.85 | 82.04 ± 6.09 | -2.87% |
| Class 2 Accuracy | 83.65 ± 7.93 | 83.12 ± 8.73 | -0.64% |
| Class 3 Accuracy | 87.87 ± 4.88 | 89.99 ± 5.88 | +2.40% |
| Class 4 Accuracy | 90.57 ± 3.55 | 89.91 ± 4.67 | -0.73% |
| Class 5 Accuracy | 96.95 ± 3.36 | 96.52 ± 4.22 | -0.44% |
| Class 6 Accuracy | 89.99 ± 7.21 | 89.52 ± 6.72 | -0.53% |
| Class 7 Accuracy | 98.67 ± 1.49 | 98.12 ± 1.39 | -0.55% |
| Class 8 Accuracy | 91.59 ± 6.74 | 93.89 ± 4.70 | +2.52% |
| Class 9 Accuracy | 92.70 ± 4.50 | 94.18 ± 2.73 | +1.60% |

**Results on Indian Pines dataset**

![Indian Pines dataset validation loss](https://github.com/JoJohnny0/nn-project/blob/main/images/IndianPines%20Loss.png?raw=true)

| metric | Static Amplification | Online Data Augmentation | Δ% |
|-|-|-|-|
| Overall Accuracy | 76.43 ± 1.71 | 77.84 ± 2.30 | +1.85% |
| Average Accuracy | 86.83 ± 1.06 | 87.45 ± 1.24 | +0.71% |
| Kappa | 73.51 ± 1.86 | 75.08 ± 2.51 | +2.14% |
| Class 1 Accuracy | 100.00 ± 0.00 | 100.00 ± 0.00 | +0.00% |
| Class 2 Accuracy | 64.86 ± 11.73 | 67.98 ± 9.03 | +4.80% |
| Class 3 Accuracy | 71.80 ± 8.86 | 76.00 ± 4.19 | +5.84% |
| Class 4 Accuracy | 97.39 ± 3.55 | 94.95 ± 6.31 | -2.50% |
| Class 5 Accuracy | 79.08 ± 9.33 | 80.77 ± 8.74 | +2.13% |
| Class 6 Accuracy | 94.21 ± 3.85 | 93.54 ± 3.06 | -0.71% |
| Class 7 Accuracy | 100.00 ± 0.00 | 100.00 ± 0.00 | +0.00% |
| Class 8 Accuracy | 98.55 ± 1.94 | 98.44 ± 2.85 | -0.11% |
| Class 9 Accuracy | 100.00 ± 0.00 | 100.00 ± 0.00 | +0.00% |
| Class 10 Accuracy | 69.73 ± 5.22 | 73.54 ± 7.27 | +5.47% |
| Class 11 Accuracy | 64.14 ± 9.44 | 66.02 ± 9.16 | +2.93% |
| Class 12 Accuracy | 66.07 ± 9.69 | 63.25 ± 12.88 | -4.27% |
| Class 13 Accuracy | 97.16 ± 3.62 | 98.11 ± 2.75 | +0.98% |
| Class 14 Accuracy | 92.08 ± 5.07 | 91.82 ± 3.98 | -0.28% |
| Class 15 Accuracy | 96.01 ± 2.11 | 96.50 ± 2.50 | +0.51% |
| Class 16 Accuracy | 98.21 ± 1.92 | 98.21 ± 2.38 | +0.00% |

While the model obtained better results on the first dataset, as expected, it's interesting to see that online data augmentation proved to be beneficial when dealing with the Indian Pines dataset, but not the Pavia University one. A possible explanation to this can be found in the fact that, while in the latter the classes are well defined and the increased variance doesn't help the model, in the first one the classes are much more similar, making for the network easier to memorize the different noise of each patch than the useful features. By constantly changing the noise, this problem is partially avoided.

## Reflections

The combination of the CTA-net architecture with the online data augmentation may be promising when dealing with particularly difficult datasets and could be beneficial when even fewer samples are available for training.

Tuning the hyperparameters and the transformations would also increase perfomance: for example, on the Pavia University dataset, reducing the possible angle for the random rotations may allow the network to learn the relative position of walls and roofs caused by the angle of the sensor when collecting the data.

In the same way, when an image is taken from far above the subject like in the Indian Pines dataset, introducing vertical and horizontal flips would not destroy the geometry underlying the photo.

Finally, if dealing with a very small dataset or in the presence of memory constraints leading to the need of a more lightweight model, this can be achieved by replacing the subsequent convolutions in the multi-resolution CNN block with a dilated convolution. In this way, the receptive field stays the same while reducing the total model size by 1/3.

## References

- *CNN-Transformer and Channel-Spatial Attention based network for hyperspectral image classification with few samples*; C. Fu, T. Zhou, T. Guo, Q. Zhu, F. Luo, B. Du; 2025
- *Conformer: Local Features Coupling Global Representations for Visual Recognition*; Z. Peng, W. Huang, S. Gu, L. Xie, Y. Wang, J. Jiao, Q. Ye; 2021
- *Learning Hyperspectral Feature Extraction and Classification with ResNeXt Network*; D. Nyasaka, J. Wang, H. Tinega; 2020
- [Pavia Univeristy HSI Dataset](https://www.kaggle.com/datasets/syamkakarla/pavia-university-hsi)
- [Indian Pines Hyperspectral Dataset](https://www.kaggle.com/datasets/abhijeetgo/indian-pines-hyperspectral-dataset)

## Reproducibility Instructions

**Dependencies:**
- Python 3.12+
- ipykernel, kagglehub, lightning, matplotlib, nbformat, numpy, scikit-learn, scipy, torch, torchmetrics, torchvision, wandb.

To reproduce the results, run the code below. The used seeds are those from 0 to 9.

Being both the models and the datasets very lightweight, the code will also run on CPU.

**Note:** depending on the hardware used, it may be needed to change the `precision` parameter in the [training section](#training) to `'32-true'` or `'16-mixed'`. This may lead to slightly different results.

## Code

### Colab Only

In [ ]:
# Install repository
!git clone https://github.com/JoJohnny0/nn-project.git

In [ ]:
# Change working directory
import os
os.chdir('nn-project')

### Setup

In [ ]:
# Install dependencies
%pip install -r requirements.txt

In [ ]:
from typing import Literal

import kagglehub
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from matplotlib.axes import Axes
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.io import loadmat
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
from torch.utils.data import DataLoader
from torchmetrics.functional import accuracy, cohen_kappa
import wandb

from modules.cta_net.cta_net import CTA_Lightning
from modules.data.loader import get_loaders

In [ ]:
# Configuration
dataset: Literal['PaviaUniversity', 'IndianPines'] = 'PaviaUniversity'
online_augmentation: bool = True
seed: int|None = None

In [ ]:
# Data Hyperparameters
patch_size: int = 15
train_samples_per_class: int = 10
val_samples_per_class: int = 5
sigma: float = 0.1
central_region_size: int = 3

# Model Hyperparameters
hidden_channels: int = 128
heads: int = 2

# Training Hyperparameters
dropout: float = 0.1
lr: float = 8e-5
batch_size: int = 32
epochs: int = 200 if online_augmentation else 150

In [ ]:
# Set random seed
if seed is not None:
    pl.seed_everything(seed)

### Data Handling

In [ ]:
# Download dataset if needed
image: NDArray[np.uint16]
labels: NDArray[np.uint8]
if dataset == 'PaviaUniversity':
    dataset_path: str = kagglehub.dataset_download('syamkakarla/pavia-university-hsi')
    image = loadmat(f'{dataset_path}/PaviaU.mat')['paviaU']
    labels = loadmat(f'{dataset_path}/PaviaU_gt.mat')['paviaU_gt']
else:
    dataset_path: str = kagglehub.dataset_download('abhijeetgo/indian-pines-hyperspectral-dataset')
    image  = np.load(f'{dataset_path}/indianpinearray.npy')
    labels = np.load(f'{dataset_path}/IPgt.npy')

In [ ]:
# Get data loaders
train_loader: DataLoader[list[torch.Tensor]]
val_loader: DataLoader[list[torch.Tensor]]
test_loader: DataLoader[list[torch.Tensor]]
train_loader, val_loader, test_loader = get_loaders(image,
                                                    labels,
                                                    patch_size,
                                                    train_samples_per_class = train_samples_per_class,
                                                    val_samples_per_class = val_samples_per_class,
                                                    online_augmentation = online_augmentation,
                                                    sigma = sigma,
                                                    central_region_size = central_region_size,
                                                    batch_size = batch_size
                                                    )

### Training

In [ ]:
# Initialize logger
wandb_logger: WandbLogger = WandbLogger(project = f"Hybrid Models for Hyperspectral Image Classification",
                                        name = f"CTA-net_{dataset}_seed={seed}",
                                        save_dir = 'out',
                                        # Parameters not logged by the trainer
                                        config = {'train_samples_per_class': train_samples_per_class,
                                                  'val_samples_per_class': val_samples_per_class,
                                                  'online': online_augmentation,
                                                  'patch_size': patch_size,
                                                  'sigma': sigma,
                                                  'central_region_size': central_region_size,
                                                  'batch_size': batch_size
                                                  }
                                        )
wandb_logger.experiment.define_metric('*', step_metric = 'epoch')

In [ ]:
# Add best checkpoint callback
save_best: ModelCheckpoint = ModelCheckpoint(monitor = 'val_loss',
                                             dirpath = f'out/checkpoints/{dataset}',
                                             filename = f'cta-net-epoch={{epoch}}-seed={seed}',
                                             save_weights_only = True
                                             )

In [ ]:
# Initialize model and trainer
n_classes: int = int(labels.max())
model: CTA_Lightning = CTA_Lightning(in_channels = image.shape[2],
                                     hidden_channels = hidden_channels,
                                     out_channels = n_classes,
                                     heads = heads,
                                     window_size = patch_size,
                                     dropout = dropout,
                                     lr = lr
                                     )
trainer: pl.Trainer = pl.Trainer(max_epochs = epochs,
                                 precision = 'bf16-mixed',
                                 callbacks = save_best,
                                 logger = wandb_logger,
                                 log_every_n_steps = len(train_loader)
                                 )

In [ ]:
# Train the model
trainer.fit(model, train_loader, val_loader)

### Test

In [ ]:
# Get predictions
preds: torch.Tensor = torch.concat(trainer.predict(model, test_loader, ckpt_path = 'best'))  # type: ignore

# Get targets
targets: torch.Tensor = torch.concat([batch[1] for batch in test_loader])

In [ ]:
# General metrics
metrics: dict[str, float] = {'Overall Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'micro', num_classes = n_classes).item(),
                             'Average Accuracy': accuracy(preds, targets, task = 'multiclass', average = 'macro', num_classes = n_classes).item(),
                             'Kappa': cohen_kappa(preds, targets, task = 'multiclass', num_classes = n_classes).item()
                             }

# Class-wise accuracy
conf_matrix: NDArray[np.int64] = confusion_matrix(targets, preds)
for i, class_acc in enumerate(conf_matrix.diagonal() / conf_matrix.sum(axis = 1)):
    metrics[f'Class {i + 1} Accuracy'] = class_acc.item()

In [ ]:
# Print metrics
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Display confusion matrix
fig: Figure
ax: Axes
fig, ax = plt.subplots(figsize = (6, 6))
disp: ConfusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix = conf_matrix)
disp.plot(ax = ax)
ax.set_title("Confusion Matrix")
fig.tight_layout()

In [ ]:
# Log to wandb
wandb_logger.experiment.log(metrics)
wandb_logger.experiment.log({'Confusion Matrix': wandb.Image(fig)})
wandb_logger.experiment.finish()
plt.close(fig)